# QLoRA Fine-Tuning: Linux CLI Assistant on TinyLlama 1.1B
**Stack:** TinyLlama-1.1B + 4-bit QLoRA (PEFT) + Hugging Face Transformers


In [1]:
# ── Cell 1: Install dependencies ──────────────────────────────────────────────
!pip install -q -U transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 22.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 15.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 13.6 MB/s eta 0:00:00


In [2]:
# ── Cell 2: Upload your dataset ───────────────────────────────────────────────
from google.colab import files
uploaded = files.upload()

Saving linux_cli_dataset.jsonl to linux_cli_dataset.jsonl


In [3]:
# ── Cell 3: Load and format dataset ───────────────────────────────────────────
from datasets import load_dataset

dataset = load_dataset('json', data_files='linux_cli_dataset.jsonl', split='train')

def format_alpaca(example):
    """Convert Alpaca-format pairs into a single prompt string TinyLlama trains on."""
    if example['input']:
        prompt = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Input:\n{example['input']}\n\n"
            f"### Response:\n{example['output']}"
        )
    else:
        prompt = (
            f"### Instruction:\n{example['instruction']}\n\n"
            f"### Response:\n{example['output']}"
        )
    return {'text': prompt}

dataset = dataset.map(format_alpaca)
print(f'Dataset size: {len(dataset)} examples')
print('\nSample prompt:')
print(dataset[0]['text'][:300])

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/68 [00:00<?, ? examples/s]

Dataset size: 68 examples

Sample prompt:
### Instruction:
How do I list all files including hidden ones in a directory?

### Response:
Use `ls -la`. The `-l` flag gives a detailed list (permissions, size, date), and `-a` shows hidden files (those starting with a dot like `.bashrc`). Example:

```
ls -la /home/user
```

You'll see entries l


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map={'': 0},
)

# --- THE NEW CRITICAL FIX ---
# This forces the Trainer to see the model as fp16, stopping it from
# reverting to bfloat16 during trainer.train()
model.config.torch_dtype = torch.float16
# ----------------------------

model.config.use_cache = False
model.config.pretraining_tp = 1

print(f'Model loaded. dtype: {model.dtype}')
print(f'GPU memory used: {torch.cuda.memory_allocated()/1e9:.2f} GB')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:213: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model loaded. dtype: torch.float16
GPU memory used: 0.78 GB


In [5]:
# ── Cell 5: Attach LoRA adapters ──────────────────────────────────────────────
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,                     # rank — higher = more capacity, more memory
    lora_alpha=32,            # scaling factor (keep 2x rank)
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj'],
    lora_dropout=0.05,
    bias='none',
    task_type='CAUSAL_LM',
)

model = get_peft_model(model, lora_config)

# --- THE FIX ---
# Hunt down any bfloat16 parameters that PEFT secretly initialized
# and convert them to float32 so the T4's GradScaler doesn't crash.
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)
# ---------------

model.print_trainable_parameters()
# Expect: ~0.4% of params trainable — that's the point of LoRA

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


In [6]:
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir='./tinyllama-linux-cli',
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    warmup_steps=10,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=10,
    save_strategy='epoch',
    optim='paged_adamw_8bit',
    lr_scheduler_type='cosine',
    report_to='none',
    dataset_text_field='text',
    max_length=512, # <-- Changed from max_seq_length to max_length
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,   # replaces 'tokenizer' in 0.9+
)

print('Trainer configured. Ready to train.')

Adding EOS to train dataset:   0%|          | 0/68 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/68 [00:00<?, ? examples/s]

Trainer configured. Ready to train.


In [7]:
# ── Cell 7: Train ─────────────────────────────────────────────────────────────

# The ultimate safety net: Sweep for any rogue BFloat16 parameters
# that the Trainer secretly initialized and force them to float32
for name, param in model.named_parameters():
    if param.dtype == torch.bfloat16:
        param.data = param.data.to(torch.float32)

# Expected time on T4: ~15-25 min for 3 epochs over 68 examples
trainer.train()
print('Training complete!')

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,1.910308


Training complete!


In [8]:
# ── Cell 8: Save adapter weights ──────────────────────────────────────────────
ADAPTER_DIR = './tinyllama-linux-cli-adapter'
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print(f'Adapter saved to {ADAPTER_DIR}')

import os
files_saved = os.listdir(ADAPTER_DIR)
print(f'Files: {files_saved}')

Adapter saved to ./tinyllama-linux-cli-adapter
Files: ['tokenizer_config.json', 'tokenizer.json', 'chat_template.jinja', 'adapter_config.json', 'README.md', 'adapter_model.safetensors']


In [10]:
# ── Cell 9: Quick inference test ──────────────────────────────────────────────
import torch

# --- THE FIX: Transition from Training to Inference ---
model.gradient_checkpointing_disable()  # Turn off training memory tricks
model.config.use_cache = True           # Turn ON inference memory cache
model.eval()                            # Lock the weights (evaluation mode)
# ----------------------------------------------------

def ask(question, max_new_tokens=200):
    prompt = f'### Instruction:\n{question}\n\n### Response:\n'
    inputs = tokenizer(prompt, return_tensors='pt').to('cuda')

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.2,
            pad_token_id=tokenizer.eos_token_id,
        )
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return response.split('### Response:')[-1].strip()

# Test it
test_questions = [
    'How do I delete a file in Linux?',
    'What does grep do?',
    'How do I check disk space?',
]

for q in test_questions:
    print(f'Q: {q}')
    print(f'A: {ask(q)}')
    print('---')

[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Q: How do I delete a file in Linux?


/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/cuda/ops.py:468: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)
[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: ```
# rm filename.ext
```
---
Q: What does grep do?


[transformers] Both `max_new_tokens` (=200) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


A: No matches

### Inst
---
Q: How do I check disk space?
A: `du -sh` (or `find /path/to/directory -maxdepth 1 -type f -
---


In [ ]:
# ── Cell 10: Download adapter weights ─────────────────────────────────────────
# Zip and download so you can use them for the merge+GGUF step
!zip -r tinyllama-linux-cli-adapter.zip tinyllama-linux-cli-adapter/

from google.colab import files
files.download('tinyllama-linux-cli-adapter.zip')
print('Download triggered. Check your browser downloads.')